# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

✔ All dependencies installed!
✔ Models will auto-download on first use (may take 2-3 minutes per model)


In [1]:
# ── CELL 3: Clone project from GitHub ──────────────────────
!git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git urdu-pipeline
%cd urdu-pipeline

# Pull latest changes
!git pull origin main

print('✔ Repository cloned and updated successfully!')
!ls -la

fatal: destination path 'urdu-pipeline' already exists and is not an empty directory.
/content/urdu-pipeline
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
Already up to date.
✔ Repository cloned and updated successfully!
total 60
drwxr-xr-x 9 root root 4096 Jun 30 12:25 .
drwxr-xr-x 1 root root 4096 Jun 30 11:13 ..
drwxr-xr-x 2 root root 4096 Jun 30 11:16 audio
-rw-r--r-- 1 root root 3861 Jun 30 12:25 config.py
drwxr-xr-x 8 root root 4096 Jun 30 13:44 .git
-rw-r--r-- 1 root root 9415 Jun 30 11:13 main.py
drwxr-xr-x 2 root root 4096 Jun 30 13:36 notebooks
drwxr-xr-x 8 root root 4096 Jun 30 11:17 outputs
drwxr-xr-x 3 root root 4096 Jun 30 13:36 pipeline
drwxr-xr-x 2 root root 4096 Jun 30 12:26 __pycache__
-rw-r--r-- 1 root root 3756 Jun 30 11:13 README.md
-rw-r--r-- 1 root root 1053 Jun 30 11:13 requirements.txt
drwxr-xr-x 4 root root 4096 Jun 30 12:05 urdu-pipeline


In [ ]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os

os.makedirs('audio', exist_ok=True)

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

# Trim from 137 seconds, duration 55 minutes (3163 seconds)
!ffmpeg -y -i audio/full.mp3 -ss 137 -t 3163 -c copy audio/test_audio.mp3

audio_path = "audio/test_audio.mp3"

size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')
print(f'   Duration: 55 minutes (starting from 137 seconds)')


[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] Destination: audio/full.webm
[download] 100% of   63.59MiB in 00:00:01 at 41.95MiB/s;33m00:000m
[ExtractAudio] Destination: audio/full.mp3
Deleting original file audio/full.webm (pass -k to keep)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfrib

In [4]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.75
  Chunk size       : 500 chars


In [1]:
# Check if audio file exists and its size
import os

audio_path = "audio/test_audio.mp3"
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    size_mins = size_mb / 0.64  # Rough conversion: 1 min ≈ 0.64 MB at 128kbps
    print(f'✔ Audio file exists: {size_mb:.1f} MB ({size_mins:.0f} minutes)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')

# Check GPU status
import torch
if torch.cuda.is_available():
    print(f'✔ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('✗ GPU not available!')

✔ Audio file exists: 45.9 MB (72 minutes)
✔ GPU: Tesla T4
   VRAM: 15.6 GB
   Used: 0.00 GB


In [2]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])

ModuleNotFoundError: No module named 'pipeline'

In [6]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF URDU TRANSCRIPT
  Interview ID      : test_audio
  Total segments    : 2347
  Confidence threshold: 0.75
    [⚠ FLAGGED] seg 001 | conf=0.70 | اسلام علیکم Dr. Majda....
    [✔] seg 002 | conf=0.75 | Welcome to the show....
    [✔] seg 003 | conf=0.99 | Thank you so much for coming today...
    [✔] seg 004 | conf=0.98 | and taking out the time....
    [⚠ FLAGGED] seg 005 | conf=0.59 | If you give us a small introduction...
    [⚠ FLAGGED] seg 006 | conf=0.67 | so that our audience who are listening to you...
    [✔] seg 007 | conf=0.76 | they know that what a brilliant...
    [✔] seg 008 | conf=0.92 | Academian you are....
    [✔] seg 009 | conf=0.97 | Thank you so much Zainab...
    [✔] seg 010 | conf=0.88 | for inviting me on your show....
    [✔] seg 011 | conf=0.89 | My name is Majda Kazmi....
    [✔] seg 012 | conf=0.84 | I am a PhD doctor....
    [⚠ FLAGGED] seg 013 | conf=0.70 | MBBS....
    [⚠ FLAGGED] seg 014 | conf=0.00 | Exactly....
    [⚠ FLAGGED]

In [7]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Source lang    : urd_Arab (Urdu)
  Target lang    : eng_Latn (English)
  Chunk size     : 500 chars

  Loading NLLB-200 model (first run downloads ~1.2GB)...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/2347 | UR: اسلام علیکم Dr. Majda....
             | EN: This is Dr. Majda....
    ✔ seg 002/2347 | UR: Welcome to the show....
             | EN: Welcome to the show....
    ✔ seg 003/2347 | UR: Thank you so much for coming today...
             | EN: thank you so much for coming today...
    ✔ seg 004/2347 | UR: and taking out the time....
             | EN: today and taking the time....
    ✔ seg 005/2347 | UR: If you give us a small introduction...
             | EN: for coming today and taking the time....
    ✔ seg 006/2347 | UR: so that our audience who are listening t...
             | EN: so that our audience who are listening to you...
    ✔ seg 007/2347 | UR: they know that what a brilliant...
             | EN: you they know what a brilliant...
    ✔ seg 008/2347 | UR: Academian you are....
             | EN: Academician you are....
    ✔ seg 009/2347 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    ✔

In [8]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID   : test_audio
  Total segments : 2347
    [✔] seg 001
             EN: This is Dr. Majda....
    [✔] seg 002
             EN: Welcome to the show....
    [✔] seg 003
             EN: thank you so much for coming today...
    [✔] seg 004
             EN: today and taking the time....
    [✔] seg 005
             EN: for coming today and taking the time....
    [✔] seg 006
             EN: so that our audience who are listening to you...
    [✔] seg 007
             EN: you they know what a brilliant...
    [⚠ FLAGGED] seg 008 | Detected language is 'es', expected 'en'
             EN: Academician you are....
    [⚠ FLAGGED] seg 009 | Detected language is 'sw', expected 'en'
             EN: Thank you so much Zainab...
    [✔] seg 010
             EN: for inviting me to your show....
    [⚠ FLAGGED] seg 011 | Detected language is 'sw', expected 'en'
             EN: My name is Majda Kazmi...
    [⚠ FLAGGED] seg 012 | Detecte

In [9]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...


  ✔ Presidio loaded.

  De-identifying full English text...



  De-identifying segments:
    [🔒] seg 001 | 1 entities removed | This is Dr. [NAME]....
    [✔] seg 002 | 0 entities removed | Welcome to the show....
    [✔] seg 003 | 0 entities removed | thank you so much for coming today...
    [🔒] seg 004 | 1 entities removed | [DATE] and taking the time....
    [🔒] seg 005 | 1 entities removed | for coming [DATE] and taking the time....
    [✔] seg 006 | 0 entities removed | so that our audience who are listening to you...
    [✔] seg 007 | 0 entities removed | you they know what a brilliant...
    [✔] seg 008 | 0 entities removed | [TRANSLATION FAILED] Academician you are....
    [🔒] seg 009 | 1 entities removed | [TRANSLATION FAILED] Thank you so much [NAME]...
    [✔] seg 010 | 0 entities removed | for inviting me to your show....
    [🔒] seg 011 | 1 entities removed | [TRANSLATION FAILED] My name is [NAME]...
    [🔒] seg 012 | 1 entities removed | [TRANSLATION FAILED] [NAME]. I'm a PhD doctor....
    [✔] seg 013 | 0 entities removed | MBBS.

    [✔] seg 017 | 0 entities removed | my son has been told...
    [✔] seg 018 | 0 entities removed | that my mother is a doctor...
    [✔] seg 019 | 0 entities removed | so he said...
    [✔] seg 020 | 0 entities removed | that she will put a medical camp here....
    [✔] seg 021 | 0 entities removed | a medical camp here....
    [✔] seg 022 | 0 entities removed | no. She's a PhD doctor....
    [✔] seg 023 | 0 entities removed | doctor. She's an academic doctor....
    [✔] seg 024 | 0 entities removed | So I have done a PhD...
    [✔] seg 025 | 0 entities removed | in my area of specialization...
    [✔] seg 026 | 0 entities removed | [TRANSLATION FAILED] and digital system design....
    [✔] seg 027 | 0 entities removed | I am an associate professor...
    [✔] seg 028 | 0 entities removed | at NAD University...
    [✔] seg 029 | 0 entities removed | in the computer engineering department....
    [✔] seg 030 | 0 entities removed | and beyond that...


    [✔] seg 031 | 0 entities removed | that there's a STEM center...
    [✔] seg 032 | 0 entities removed | which is...
    [✔] seg 033 | 0 entities removed | I think...
    [✔] seg 034 | 0 entities removed | is one of a kind....
    [✔] seg 035 | 0 entities removed | in any higher education institute...
    [✔] seg 036 | 0 entities removed | the best of my knowledge...
    [✔] seg 037 | 0 entities removed | this is the first STEM center...
    [✔] seg 038 | 0 entities removed | I am the founding director....
    [✔] seg 039 | 0 entities removed | the founding director....
    [✔] seg 040 | 0 entities removed | [TRANSLATION FAILED] that I'm a co-principal investigator...
    [✔] seg 041 | 0 entities removed | of the National Center of Artificial Intelligence...
    [✔] seg 042 | 0 entities removed | which I lead to the Artificial Intelligence...
    [✔] seg 043 | 0 entities removed | and Computer Vision site....
    [✔] seg 044 | 0 entities removed | Computer Vision site....
    [✔] se

...

    [✔] seg 2283 | 0 entities removed | to someone...
    [✔] seg 2284 | 0 entities removed | to someone...
    [✔] seg 2285 | 0 entities removed | to someone...
    [✔] seg 2286 | 0 entities removed | to someone...
    [✔] seg 2287 | 0 entities removed | to someone...
    [✔] seg 2288 | 0 entities removed | to someone...
    [✔] seg 2289 | 0 entities removed | to someone...
    [✔] seg 2290 | 0 entities removed | to someone...
    [✔] seg 2291 | 0 entities removed | to someone...
    [✔] seg 2292 | 0 entities removed | to someone...
    [✔] seg 2293 | 0 entities removed | to someone...
    [✔] seg 2294 | 0 entities removed | to someone...
    [✔] seg 2295 | 0 entities removed | to someone...
    [✔] seg 2296 | 0 entities removed | to someone...
    [✔] seg 2297 | 0 entities removed | to someone...
    [✔] seg 2298 | 0 entities removed | to someone...
    [✔] seg 2299 | 0 entities removed | to someone...


    [✔] seg 2300 | 0 entities removed | to someone...
    [✔] seg 2301 | 0 entities removed | to someone...
    [✔] seg 2302 | 0 entities removed | to someone...
    [✔] seg 2303 | 0 entities removed | to someone...
    [✔] seg 2304 | 0 entities removed | to someone...
    [✔] seg 2305 | 0 entities removed | to someone...
    [✔] seg 2306 | 0 entities removed | to someone...
    [✔] seg 2307 | 0 entities removed | to someone...
    [✔] seg 2308 | 0 entities removed | to someone...
    [✔] seg 2309 | 0 entities removed | to someone...
    [✔] seg 2310 | 0 entities removed | to someone...
    [✔] seg 2311 | 0 entities removed | to someone...
    [✔] seg 2312 | 0 entities removed | to someone...


    [✔] seg 2313 | 0 entities removed | to someone...
    [✔] seg 2314 | 0 entities removed | to someone...
    [✔] seg 2315 | 0 entities removed | to someone...
    [✔] seg 2316 | 0 entities removed | to someone...
    [✔] seg 2317 | 0 entities removed | to someone...
    [✔] seg 2318 | 0 entities removed | to someone...
    [✔] seg 2319 | 0 entities removed | to someone...
    [✔] seg 2320 | 0 entities removed | to someone...
    [✔] seg 2321 | 0 entities removed | to someone...
    [✔] seg 2322 | 0 entities removed | to someone...
    [✔] seg 2323 | 0 entities removed | to someone...
    [✔] seg 2324 | 0 entities removed | to someone...
    [✔] seg 2325 | 0 entities removed | to someone...
    [✔] seg 2326 | 0 entities removed | to someone...


    [✔] seg 2327 | 0 entities removed | to someone...
    [✔] seg 2328 | 0 entities removed | to someone...
    [✔] seg 2329 | 0 entities removed | to someone...
    [✔] seg 2330 | 0 entities removed | to someone...
    [✔] seg 2331 | 0 entities removed | to someone...
    [✔] seg 2332 | 0 entities removed | to someone...
    [✔] seg 2333 | 0 entities removed | to someone...
    [✔] seg 2334 | 0 entities removed | to someone...
    [✔] seg 2335 | 0 entities removed | to someone...
    [✔] seg 2336 | 0 entities removed | to someone...
    [✔] seg 2337 | 0 entities removed | to someone...
    [✔] seg 2338 | 0 entities removed | to someone...
    [✔] seg 2339 | 0 entities removed | to someone...
    [✔] seg 2340 | 0 entities removed | to someone...
    [✔] seg 2341 | 0 entities removed | to someone...


    [✔] seg 2342 | 0 entities removed | to someone...
    [✔] seg 2343 | 0 entities removed | to someone...
    [✔] seg 2344 | 0 entities removed | to someone...
    [✔] seg 2345 | 0 entities removed | to someone...
    [✔] seg 2346 | 0 entities removed | to someone...
    [✔] seg 2347 | 0 entities removed | to someone...

  ── De-identification Summary ─────────────────
  Total entities removed : 9
  Unique entities        : 6
    PERSON                    → 5 removed
    DATE_TIME                 → 3 removed
    LOCATION                  → 1 removed
  ✔ Saved → /content/urdu-pipeline/outputs/5_deidentified/test_audio_deidentified.json

  ✔ Stage 5 complete.

Entities removed: 9

── De-identified Text Preview (first 500 chars) ──
This is Dr. [NAME]. Welcome to the show. thank you so much for coming [DATE] and taking the time. for coming [DATE] and taking the time. so that our audience who are listening to you you they know what a brilliant [TRANSLATION FAILED] Academician you are. [TR

In [ ]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')

In [ ]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
from google.colab import files

interview_id = stage1_result['interview_id']

# Zip the entire outputs folder
zip_name = f'{interview_id}_outputs.zip'
shutil.make_archive(
    base_name=interview_id + '_outputs',
    format='zip',
    root_dir='urdu-pipeline',
    base_dir='outputs'
)

print(f'✔ Zipped outputs as {zip_name}')
print('Downloading...')
files.download(zip_name)

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import sys
sys.path.insert(0, 'urdu-pipeline')

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

# Change this to your audio path
run_pipeline('urdu-pipeline/audio/your_audio.mp3', start_stage=1)